# Notebook 5 - Predictive Maintenance (Demo)

Run the four labs of Chapter 5 in Python: RUL from sensors, degradation and thresholds, early warning from residuals, and the cost of decisions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-05-predictive-maintenance.ipynb)

- Lecture: [Chapter 5 - Predictive Maintenance with Linear Models](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-05-predictive-maintainance.md)
- Labs: [LM501](https://rathachai.github.io/DA-LAB/learnings/linear/lm501.html) · [LM502](https://rathachai.github.io/DA-LAB/learnings/linear/lm502.html) · [LM503](https://rathachai.github.io/DA-LAB/learnings/linear/lm503.html) · [LM504](https://rathachai.github.io/DA-LAB/learnings/linear/lm504.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## Setup
We use only numpy, pandas, matplotlib and scikit-learn. Run the cells from top to bottom. Each cell shows one result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
pd.set_option("display.precision", 2)
plt.rcParams["figure.figsize"] = (8, 3.5)
print("ready")

# PART 1 - RUL from sensors (LM501)

**RUL** = Remaining Useful Life, the hours a machine has left. Here `y` is the RUL, and `s1`..`s7` are sensors. 60 machines, 40 snapshots each.

### Step 1.1 - Load the data

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm501_data.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

### Step 1.2 - Summary statistics
Look at the scales: the sensors have very different units, so we will normalise them for plotting.

In [ ]:
sensors = ["s1", "s2", "s3", "s4", "s5", "s6", "s7"]
df[sensors + ["y"]].describe().T

### Step 1.3 - One machine over time
We normalise every column to 0-1 (min-max) so they fit on one plot. The thick black line is the RUL: it goes down as the machine wears.

In [ ]:
m1 = df[df.machine == 1].reset_index(drop=True)
norm = (m1[sensors + ["y"]] - m1[sensors + ["y"]].min()) / (m1[sensors + ["y"]].max() - m1[sensors + ["y"]].min())
ax = norm[sensors].plot(lw=1)
norm["y"].plot(ax=ax, color="k", lw=3, label="RUL (y)")
ax.set_xlabel("snapshot number"); ax.set_ylabel("normalised value"); ax.legend(ncol=4, fontsize=8)
plt.show()
norm.head()

### Step 1.4 - Correlation of each sensor with RUL
Strong correlation (near +1 or -1) means a useful sensor. Near 0 means no *straight-line* relation: noise, or a curve.

In [ ]:
corr = df[sensors].corrwith(df["y"]).rename("corr_with_y").to_frame()
corr["abs_corr"] = corr["corr_with_y"].abs()
display(corr.sort_values("abs_corr", ascending=False))
corr["corr_with_y"].plot.bar(color="tab:blue"); plt.axhline(0, color="k", lw=0.8); plt.ylabel("r"); plt.show()

### Step 1.5 - Log transform of s7
`s7` grows exponentially with wear (like wear debris). The log turns an exponential curve into a straight line, so the correlation improves.

In [ ]:
df["log_s7"] = np.log(df["s7"])
print("corr(s7, y)     =", round(df["s7"].corr(df["y"]), 3))
print("corr(log_s7, y) =", round(df["log_s7"].corr(df["y"]), 3))
df[["id", "machine", "s7", "log_s7", "y"]].head()

### Step 1.6 - Fit a model: split by row vs split by machine
We keep the good sensors (`s1, s2, s4, s6, log_s7`) and fit a `LinearRegression`.

- **By row**: random rows go to the test set, so the test machines were already seen in training.
- **By machine**: whole machines are held out, like new machines in the field.

The score is the **test MAE** (mean absolute error, in hours).

In [ ]:
feats = ["s1", "s2", "s4", "s6", "log_s7"]
X, y, g = df[feats], df["y"], df["machine"]
def make_model(): return make_pipeline(StandardScaler(), LinearRegression())
def test_mae(split, seed):
    if split == "by row":
        tr, te = train_test_split(df.index, test_size=0.25, random_state=seed)
    else:
        tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(X, y, g))
        tr, te = df.index[tr], df.index[te]
    return mean_absolute_error(y[te], make_model().fit(X.loc[tr], y[tr]).predict(X.loc[te]))
pd.DataFrame({"split": ["by row", "by machine"], "test MAE (h)": [test_mae("by row", 0), test_mae("by machine", 0)]})

### Step 1.7 - Repeat over 20 random splits
One split can be lucky. We repeat 20 times and compare the distributions. We also add a more flexible model (polynomial degree 3), which can memorise machine fingerprints more easily.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
def run(model_fn, name):
    rows = []
    for seed in range(20):
        for split in ["by row", "by machine"]:
            if split == "by row":
                tr, te = train_test_split(df.index, test_size=0.25, random_state=seed)
            else:
                a, b = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(X, y, g))
                tr, te = df.index[a], df.index[b]
            mae = mean_absolute_error(y[te], model_fn().fit(X.loc[tr], y[tr]).predict(X.loc[te]))
            rows.append({"model": name, "split": split, "seed": seed, "MAE": mae})
    return pd.DataFrame(rows)
res = pd.concat([run(make_model, "linear"),
                 run(lambda: make_pipeline(StandardScaler(), PolynomialFeatures(3), LinearRegression()), "poly3")])
res.groupby(["model", "split"])["MAE"].agg(["mean", "std", "min", "max"])

In [ ]:
res.boxplot(column="MAE", by=["model", "split"], figsize=(8, 3.5))
plt.suptitle(""); plt.title("Test MAE over 20 random splits"); plt.ylabel("MAE (h)"); plt.show()

**Reading the result.** The by-row MAE is a little lower (better-looking) than the by-machine MAE, because the by-row test set shares machines with the training set. For this simple straight-line model on these sensors the gap is small (a fraction of an hour); it is larger for the more flexible model, and in real plants with strong machine fingerprints it can be big. The honest number for *new* machines is the by-machine one.

# PART 2 - Degradation and threshold (LM502)

A **health index** goes from 100 (new) down to 0 (failed). We fit a straight line to recent readings and extrapolate to a threshold **theta** (repair) and to 0 (failure).

### Step 2.1 - Load the five degradation patterns

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm502_degradation.csv"
deg = pd.read_csv(url)
print(deg.shape)
deg.head()

### Step 2.2 - True health vs sensor reading
Grey = true health, blue = what the sensor reports. The dashed line is the real failure time.

In [ ]:
patterns = deg["pattern"].unique()
fig, axs = plt.subplots(2, 3, figsize=(11, 5), sharey=True)
for ax, p in zip(axs.flat, patterns):
    s = deg[deg.pattern == p]
    ax.plot(s.t, s.health_sensor, ".", ms=2, color="tab:blue"); ax.plot(s.t, s.health_true, color="gray")
    ax.axvline(s.failure_time.iloc[0], ls="--", color="r"); ax.set_title(p)
axs.flat[-1].axis("off"); plt.tight_layout(); plt.show()
deg.groupby("pattern")["failure_time"].first().to_frame()

### Step 2.3 - Predict failure for one pattern
We pick pattern `linear`, a "now" time of 120 h, and fit a line to the last **L = 30** sensor readings. Then (formulas from the lecture):

- repair time: `t_theta = (theta - b0) / b1`
- failure time: `t_fail = (0 - b0) / b1`
- RUL = `t_fail - now`

In [ ]:
THETA, L = 50, 30
def predict_failure(s, now, L=L, theta=THETA):
    w = s[(s.t <= now) & (s.t > now - L)]
    lr = LinearRegression().fit(w[["t"]], w["health_sensor"])
    b0, b1 = lr.intercept_, lr.coef_[0]
    if b1 >= 0: return dict(b0=b0, b1=b1, t_theta=np.nan, t_fail=np.nan)
    return dict(b0=b0, b1=b1, t_theta=(theta - b0) / b1, t_fail=(0 - b0) / b1)
s = deg[deg.pattern == "linear"]; now = 120
r = predict_failure(s, now); true_fail = s.failure_time.iloc[0]
pd.Series({"slope b1 (pts/h)": r["b1"], "repair time t_theta": r["t_theta"], "predicted failure": r["t_fail"],
           "true failure": true_fail, "predicted RUL": r["t_fail"] - now, "true RUL": true_fail - now,
           "RUL error (h)": (r["t_fail"] - now) - (true_fail - now)}).to_frame("value")

### Step 2.4 - Draw the fitted line and its extrapolation
The red line is the fitted trend, extended to the threshold (orange) and to zero health.

In [ ]:
def draw(s, now, title):
    r = predict_failure(s, now); w = s[(s.t <= now) & (s.t > now - L)]
    tt = np.linspace(now - L, max(r["t_fail"], now) + 10, 100)
    plt.plot(s.t, s.health_sensor, ".", ms=3, color="tab:blue"); plt.plot(w.t, w.health_sensor, ".", color="navy")
    plt.plot(tt, r["b0"] + r["b1"] * tt, "r"); plt.axhline(THETA, color="orange"); plt.axhline(0, color="k", lw=0.5)
    plt.axvline(now, color="g", ls=":"); plt.axvline(s.failure_time.iloc[0], color="gray", ls="--")
    plt.ylim(-10, 105); plt.title(title); plt.xlabel("t (h)"); plt.ylabel("health")
draw(s, 120, "linear: 'now' = 120 h (green), true failure (grey dashed)"); plt.show()

### Step 2.5 - The straight line is optimistic for `accelerating` wear
Same recipe at the same 'now' for every pattern. A positive RUL error means the line promises *more* life than the machine really has (optimistic).

In [ ]:
now = 150
rows = []
for p in patterns:
    sp = deg[deg.pattern == p]; r = predict_failure(sp, now); tf = sp.failure_time.iloc[0]
    rows.append({"pattern": p, "pred_failure": r["t_fail"], "true_failure": tf, "RUL_error_h": r["t_fail"] - tf})
pd.DataFrame(rows).set_index("pattern")

In [ ]:
draw(deg[deg.pattern == "accelerating"], 150, "accelerating: the line (red) crosses zero too late")
plt.show()

### Step 2.6 - Predicted RUL over time
We move 'now' forward and repeat the prediction. The black line is the true RUL. A good predictor stays close to it; the closer to failure, the better the forecast.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(11, 5), sharey=True)
for ax, p in zip(axs.flat, patterns):
    sp = deg[deg.pattern == p]; tf = sp.failure_time.iloc[0]
    nows = np.arange(60, int(tf), 5)
    pred = [predict_failure(sp, n)["t_fail"] - n for n in nows]
    ax.plot(nows, tf - nows, "k", label="true RUL"); ax.plot(nows, pred, "r.-", label="predicted"); ax.set_title(p)
    ax.set_ylim(-50, 400)
axs.flat[0].legend(); axs.flat[-1].axis("off"); plt.tight_layout(); plt.show()

# PART 3 - Early warning from residuals (LM503)

Idea: learn what a **healthy** signal looks like, forecast it, and watch the **residual** (actual minus predicted). A growing residual means something changed, like a point outside the limits on a control chart.

### Step 3.1 - Load the signals

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm503_signals.csv"
sig = pd.read_csv(url)
print(sig.scenario.unique())
sig.head()

### Step 3.2 - Look at the scenarios
Red = points where the fault is active (`is_fault = 1`).

In [ ]:
scen = sig.scenario.unique()
fig, axs = plt.subplots(len(scen), 1, figsize=(9, 8), sharex=True)
for ax, sc in zip(axs, scen):
    s = sig[sig.scenario == sc]; ax.plot(s.t, s.signal, lw=0.8); ax.plot(s.t[s.is_fault == 1], s.signal[s.is_fault == 1], "r.", ms=3)
    ax.set_ylabel(sc, fontsize=8)
plt.tight_layout(); plt.show()
sig.groupby("scenario")["is_fault"].agg(["sum", lambda x: x.values.argmax()]).set_axis(["fault points", "fault starts at t"], axis=1)

### Step 3.3 - AR(w) model on the healthy start
An **AR(w)** model predicts the next value from the previous `w` values, using a `LinearRegression` on lagged columns. We train on the first **180 points** only (healthy), and take `sigma` = standard deviation of the training residuals.

In [ ]:
W, NTRAIN = 5, 180
def lag_table(x, w=W):
    t = pd.DataFrame({f"lag{i}": x.shift(i) for i in range(1, w + 1)}); t["y"] = x
    return t.dropna()
s = sig[sig.scenario == "drift"].set_index("t")
tab = lag_table(s["signal"])
train = tab[tab.index < NTRAIN]
ar = LinearRegression().fit(train.drop(columns="y"), train["y"])
tab["pred"] = ar.predict(tab.drop(columns="y")); tab["resid"] = tab["y"] - tab["pred"]
sigma = tab.loc[tab.index < NTRAIN, "resid"].std()
print("sigma =", round(sigma, 3))
tab.head()

### Step 3.4 - k-sigma alarm
We use k = 2 here (drift is slow, so k = 3 would miss it). Alarm when `|residual| > k * sigma` for **m = 3** samples in a row. We then measure:

- **detection delay** = first alarm after the fault starts, minus the fault start time;
- **false alarms** = alarm samples in the healthy test part (after training, before the fault).

In [ ]:
M_ROWS = 3
def alarms(resid, sigma, k, m=M_ROWS):
    over = (resid.abs() > k * sigma).astype(int)
    return over.rolling(m).sum() == m
def evaluate(sc, k, w=W, m=M_ROWS):
    s = sig[sig.scenario == sc].set_index("t"); tb = lag_table(s["signal"], w)
    mdl = LinearRegression().fit(tb[tb.index < NTRAIN].drop(columns="y"), tb[tb.index < NTRAIN]["y"])
    res = tb["y"] - mdl.predict(tb.drop(columns="y")); sg = res[res.index < NTRAIN].std()
    al = alarms(res, sg, k, m); fs = s.index[s.is_fault == 1].min() if s.is_fault.any() else None
    healthy = al[(al.index >= NTRAIN) & ((al.index < fs) if fs is not None else True)]
    after = al[al.index >= fs] if fs is not None else al[:0]
    delay = (after[after].index.min() - fs) if fs is not None and after.any() else np.nan
    return dict(res=res, sigma=sg, alarm=al, fault_start=fs, delay=delay, false_alarms=int(healthy.sum()))
KD = 2   # k used for this demo
e = evaluate("drift", KD)
print("fault starts:", e["fault_start"], "| detection delay:", e["delay"], "| false alarms:", e["false_alarms"])

In [ ]:
s = sig[sig.scenario == "drift"].set_index("t")
fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(s.signal, lw=0.8, label="signal"); axs[0].plot(tab.pred, lw=0.8, label="AR prediction"); axs[0].legend()
r = e["res"]; axs[1].plot(r, lw=0.8)
axs[1].axhline(KD * e["sigma"], color="orange"); axs[1].axhline(-KD * e["sigma"], color="orange")
axs[1].plot(r[e["alarm"]], "r.", label="alarm"); axs[1].set_ylabel("residual")
for a in axs: a.axvline(NTRAIN, color="g", ls=":"); a.axvline(e["fault_start"], color="gray", ls="--")
axs[1].legend(); plt.tight_layout(); plt.show()

### Step 3.5 - The trade-off: sweep k = 1..6 for all fault scenarios
Small k: early detection but false alarms. Large k: quiet but late (NaN = never detected).

In [ ]:
rows = []
for sc in ["drift", "level_shift", "growing_variance", "bearing_oscillation"]:
    for k in range(1, 7):
        e = evaluate(sc, k); rows.append({"scenario": sc, "k": k, "delay": e["delay"], "false_alarms": e["false_alarms"]})
out = pd.DataFrame(rows)
display(out.pivot(index="k", columns="scenario", values="delay").rename_axis(columns="detection delay (samples)"))
out.pivot(index="k", columns="scenario", values="false_alarms").rename_axis(columns="false alarms")

# PART 4 - Cost of decisions (LM504)

A prediction is only useful if it leads to a better decision. Wrong decisions have **asymmetric** costs: a breakdown costs far more than a planned repair.

### Step 4.1 - Load the fleet
200 machines with the true RUL and three predictors: unbiased, pessimistic (too low) and optimistic (too high).

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm504_fleet.csv"
fleet = pd.read_csv(url)
preds = ["predicted_unbiased", "predicted_pessimistic", "predicted_optimistic"]
print(fleet.shape)
fleet.head()

### Step 4.2 - Predicted vs actual
Points on the dashed y = x line would be perfect. Above the line = optimistic, below = pessimistic.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, p in zip(axs, preds):
    ax.scatter(fleet.actual_rul, fleet[p], s=8); ax.plot([0, 450], [0, 450], "k--"); ax.set_title(p); ax.set_xlabel("actual RUL (h)")
axs[0].set_ylabel("predicted RUL (h)"); plt.tight_layout(); plt.show()

### Step 4.3 - Error metrics
MAE, RMSE (typical error in hours), MAPE (% error), R2, and **bias** (mean of predicted minus actual; positive = optimistic).

In [ ]:
a = fleet["actual_rul"]
met = pd.DataFrame({p: {"MAE": mean_absolute_error(a, fleet[p]), "RMSE": mean_squared_error(a, fleet[p]) ** 0.5,
    "MAPE %": 100 * mean_absolute_percentage_error(a, fleet[p]), "R2": r2_score(a, fleet[p]),
    "bias": (fleet[p] - a).mean()} for p in preds}).T
met

### Step 4.4 - The decision rule and its cost
Costs: repair `Cp = 1000`, wasted life `Cw = 5` per hour, breakdown `Cf = 8000`.

- If predicted RUL <= **H**: repair now, cost = `Cp + Cw * actual_rul`.
- Otherwise run to the next inspection `D` hours later: if `actual_rul < D` the machine breaks down (cost `Cf`), else cost 0.

In [ ]:
CP, CW, CF, D = 1000, 5, 8000, 50
def total_cost(pred, actual, H, D=D):
    repair = pred <= H
    cost = np.where(repair, CP + CW * actual, np.where(actual < D, CF, 0.0))
    return cost.sum()
H = 100
pd.DataFrame({p: {"repairs": int((fleet[p] <= H).sum()),
                  "breakdowns": int(((fleet[p] > H) & (a < D)).sum()),
                  "total cost": total_cost(fleet[p], a, H)} for p in preds}).T

### Step 4.5 - Cost vs H for the three predictors
Every curve has a U shape: low H means breakdowns, high H means wasted life. The dot marks the best H.

In [ ]:
Hs = np.arange(0, 301, 5)
curves = pd.DataFrame({p: [total_cost(fleet[p], a, h) for h in Hs] for p in preds}, index=Hs)
curves.index.name = "H"
ax = curves.plot(figsize=(8, 3.8))
for p in preds: ax.plot(curves[p].idxmin(), curves[p].min(), "ko")
ax.set_ylabel("total cost"); plt.show()
curves.head()

### Step 4.6 - Lowest RMSE is not necessarily the cheapest
We compare the RMSE ranking with the cost ranking, each predictor at its own best H.

In [ ]:
best = pd.DataFrame({"RMSE": met["RMSE"], "bias": met["bias"], "best H": curves.idxmin(), "min total cost": curves.min()})
best["RMSE rank"] = best["RMSE"].rank().astype(int); best["cost rank"] = best["min total cost"].rank().astype(int)
display(best)
print("lowest RMSE :", best["RMSE"].idxmin()); print("cheapest    :", best["min total cost"].idxmin())

# Summary - key takeaways

- **RUL regression**: pick sensors by correlation, straighten curved sensors with a log, and judge the model on **new machines** (split by machine, not by row).
- **Degradation**: a straight line extrapolated to a threshold gives repair and failure times, but it is **optimistic** for accelerating wear. Predictions improve as the machine gets closer to failure.
- **Early warning**: train on healthy data, watch the residuals, alarm at `k * sigma`. Small k = fast but noisy, large k = quiet but late.
- **Costs**: a breakdown costs much more than a repair, so the best threshold H is cautious. The lowest-RMSE model is not always the cheapest: choose by **total cost**.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io